# D03 · 什么时候来

"什么时候"有四个尺度，本册逐个看：

| 尺度 | 变量 | 能回答什么 |
| --- | --- | --- |
| 年内 | 到达月 | 淡旺季，什么时候该加人 |
| 周内 | 到达星期 | 一周里哪天最忙 |
| 提前多久订 | `lead_time` | 客人是计划型还是临时型 |
| 住多久 | `total_nights` | 短住还是长住 |

四个尺度里，**只有提前期与取消率是单调关系**。其他三个都有例外，
所以本册的结论会反复出现"某月/某天更高，但原因不在这"。

时间口径的坑在 D01 已经说过：只有 2016 是完整年，同比只能取 1—8 月。

In [ ]:
%autoreload 2
%matplotlib inline

import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hotel_insight import config, rate, timing, viz
from hotel_insight.viz import FAINT, INK, MAIN, MAIN_LIGHT, MAIN_MID, MUTED

viz.apply_theme()
pd.set_option("display.width", 220)
pd.set_option("display.unicode.east_asian_width", True)

con = sqlite3.connect(config.DB_PATH)
s = config.load_settings()
a = s["analysis"]
NB = "D03"
SOURCE = "数据源：hotel.db / dwd_bookings（85,585 条入住记录，2015-07 至 2017-08）"

d = pd.read_sql("SELECT * FROM dwd_bookings", con)
print(f"记录数 {len(d):,}；取消率 {d['is_canceled'].mean():.2%}")
assert len(d) == 85585

## 1 年内节奏：8 月是全年峰值，11 月最淡

跨三年的月度记录数会把"数据窗口不完整"混进来，所以这一节只画 **2016 年**——
唯一完整的自然年。

In [ ]:
# ── 出图 D03_01：2016 年季节指数 ──
# 为什么用季节指数而不是记录数：记录数的绝对值受酒店规模影响，指数是"相对月均的倍数"，
# 换成别家酒店也能比；而且指数 = 1 的那条线让"高于/低于平均"一眼可见。
# 规范落在哪几行：只算完整年（timing.season_index 会拒绝非完整年）/ 均值线用灰虚线 /
#               高于均值的月份用主色、低于用浅色
# ---- 1 指标 ----
si = timing.season_index(d, 2016)
peak, low = si.idxmax(), si.idxmin()
title = f"{peak} 月是月均的 {si.max():.2f} 倍，{low} 月只有 {si.min():.2f} 倍"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = "口径：2016 年 12 个月的记录数 ÷ 该年月均；1.0 表示与平均持平"
# ---- 2 画图 ----
fig, ax = plt.subplots(figsize=(9.6, 4.0))
xs = si.index.to_numpy()
colors = [MAIN if v >= 1 else MAIN_LIGHT for v in si.to_numpy()]
ax.bar(xs, si.to_numpy(), color=colors, width=0.66, zorder=2)
ax.axhline(1.0, color=viz.GRAY_DARK, linewidth=1, linestyle=(0, (4, 3)), zorder=1)
for x, v in zip(xs, si.to_numpy()):
    ax.text(x, v + 0.015, f"{v:.2f}", ha="center", va="bottom", fontsize=8.5, color=INK)
ax.set_xticks(xs, [f"{m} 月" for m in xs], fontsize=9)
ax.set_ylabel("季节指数")
ax.set_ylim(0, si.max() * 1.18)
ax.grid(axis="x", visible=False)
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D03_01_季节指数.png")
# ---- 4 读回断言：柱高等于指数，且 12 个月全在 ----
heights = [b.get_height() for b in ax.patches]
assert np.allclose(heights, si.to_numpy()), "柱高与季节指数不一致"
assert len(heights) == 12
plt.show()

8 月 1.27、10 月 1.20 是双峰（暑期尾 + 秋季），1 月 0.52 最淡。
注意这是**记录数**的季节性，不是取消率的季节性——两个峰不是同一件事。

`timing.season_index()` 用 `raise` 拒绝非完整年。这不是洁癖：
2015 只有 7—12 月，用它算指数会把 1 月算成 0，图上就凭空多出一个"冬季停业"。

## 2 周内节奏：周一入住最多，周五取消率最高

星期几不是数据字段，是从 `arrival_date` 算出来的（DQ-05 已核对日期可靠）。

In [ ]:
# ── 出图 D03_02：周内到店量与取消率 ──
# 为什么用两个子图：到店量（供给侧要排班）与取消率（需求侧要风控）是两个问题，
# 双轴会把"周五柱子不高但折线高"读成因果。
# 规范落在哪几行：到店量按周一→周日固定顺序（有序变量，不按大小排）/ 取消率带区间 /
#               最高与最低的差距写进标题
# ---- 1 指标 ----
dow = timing.dow_profile(d)
peak_n, low_n = dow["记录数"].idxmax(), dow["记录数"].idxmin()
hi_r, lo_r = dow["取消率"].idxmax(), dow["取消率"].idxmin()
title = f"{peak_n}到店最多，{hi_r}取消率 {dow.loc[hi_r, '取消率']:.0%} 为全周最高"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = (f"取消率从 {dow['取消率'].min():.1%}（{lo_r}）到 {dow['取消率'].max():.1%}（{hi_r}），"
            f"差 {(dow['取消率'].max() - dow['取消率'].min()) * 100:.0f} 个百分点")
# ---- 2 画图 ----
fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.6))
viz.hbar(axes[0], dow.index, dow["记录数"].to_numpy(), value_fmt="{:,.0f}",
         colors=[MAIN if x == peak_n else MAIN_MID for x in dow.index])
axes[0].xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
axes[0].set_title("到店记录数（条）", fontsize=10, color=MUTED, loc="left")
viz.hbar(axes[1], dow.index, dow["取消率"].to_numpy(),
         value_fmt="{:.1%}", errors=(dow["下限"].to_numpy(), dow["上限"].to_numpy()),
         colors=[MAIN if x == hi_r else MAIN_MID for x in dow.index])
axes[1].xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
axes[1].set_xlim(0, dow["上限"].max() * 1.25)
axes[1].tick_params(labelleft=False)
axes[1].set_title("取消率（95% 区间）", fontsize=10, color=MUTED, loc="left")
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D03_02_周内节奏.png")
# ---- 4 读回断言 ----
assert list(dow.index) == ["周一", "周二", "周三", "周四", "周五", "周六", "周日"], "顺序被打乱了"
assert np.allclose([b.get_width() for b in axes[1].patches], dow["取消率"].to_numpy())
plt.show()

周一入住最多（16.2%），周二最少（12.5%）——这是商务客的典型节奏：周一到、周五走。
取消率却是周五（31.0%）、周六（30.4%）最高，周二最低（24.7%），
两者**不是同一个峰**：来的人多的时候，取消的也多不到哪去，反过来也一样。

周内的取消率差只有 6 个百分点，是全部分析里最弱的一条线索。
它更适合排班（周五到店多、周末退房多），不适合当"取消原因"。

## 3 提前期：唯一呈单调关系的变量

"提前多久订"和"会不会取消"几乎是线性关系。这是本册最重要的发现，
也是 D05 里超额取消单第二多的因素（第一是市场细分）。

In [ ]:
# ── 出图 D03_03：提前期分档的规模与取消率 ──
# 为什么分档而不是画散点：lead_time 有 700 多个取值，散点会糊成一片；
# 分档（7/30/90/180/365 天）是业务能用的口径——"什么时候该发提醒"看的就是档。
# 规范落在哪几行：档位是有序变量，按时间从短到长排，不按记录数排 / 区间 / 每档标记录数
# ---- 1 指标 ----
lt = timing.binned_profile(d, "lead_time", a["lead_time_bins"], a["lead_time_labels"])
first = a["lead_time_labels"][0]
hi_label = a["lead_time_labels"][4]                  # 181–365 天：样本够、取消率已经很高
long_share = lt.loc[a["lead_time_labels"][3]:, "占比"].sum()   # 91 天以上合计
title = f"7 天内订的取消 {lt.loc[first, '取消率']:.0%}，提前半年以上升到 {lt.loc[hi_label, '取消率']:.0%}"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = (f"共 {len(lt)} 档；7 天内占 {lt.loc[first, '占比']:.0%} 的记录，"
            f"91 天以上占 {long_share:.0%}；误差线为 95% Wilson 区间")
# ---- 2 画图 ----
fig, axes = plt.subplots(1, 2, figsize=(10.6, 3.8))
viz.hbar(axes[0], lt.index, lt["记录数"].to_numpy(), value_fmt="{:,.0f}", colors=MAIN_MID)
axes[0].xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
axes[0].set_title("记录数（条）", fontsize=10, color=MUTED, loc="left")
viz.hbar(axes[1], lt.index, lt["取消率"].to_numpy(), value_fmt="{:.1%}",
         errors=(lt["下限"].to_numpy(), lt["上限"].to_numpy()),
         colors=[MAIN if x in (a["lead_time_labels"][3], a["lead_time_labels"][4]) else MAIN_MID
                 for x in lt.index])
axes[1].xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
axes[1].set_xlim(0, lt["上限"].max() * 1.25)
axes[1].tick_params(labelleft=False)
axes[1].set_title("取消率（95% 区间）", fontsize=10, color=MUTED, loc="left")
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D03_03_提前期与取消率.png")
# ---- 4 读回断言：取消率必须随档位单调不减 ----
rates = lt["取消率"].to_numpy()
assert np.all(np.diff(rates) > 0), f"取消率不是单调递增：{rates}"
assert np.allclose([b.get_width() for b in axes[1].patches], rates)
plt.show()

从 8.4% 一路升到 41.9%，**六档里没有一次回头**。这条单调性有两个用处：

1. **可预测**：下单那一刻就知道 lead_time，可以据此给订单打风险标签；
2. **可干预**：长提前期订单的取消发生在更早（D05 会验证），还来得及再确认。

但要注意分母：365 天以上只有 547 条（0.6%），它的 41.9% 区间很宽，
不能拿它当"最危险的一档"去制定政策——真正有量的是 91—365 天那 29,206 条。

## 4 住多久：住得越久越容易取消，但幅度不大

这个维度常被当成"长住客更慎重"，数据不支持这个说法——差距只有 11 个百分点，
而且和提前期高度相关（长住往往提前订）。

In [ ]:
stay = timing.binned_profile(d, "total_nights", a["stay_bins"], a["stay_labels"])
stay.round({"占比": 3, "累计占比": 3, "取消率": 3, "下限": 3, "上限": 3})

1 晚占 37.5%、取消率 22.6%；8 晚以上只占 4.4%、取消率 33.6%。
方向一致但幅度有限，而且分档内样本量差异大（4.4% 那一档只有 3,746 条）。
本册把它当"辅助变量"，不当主线索。

## 5 同比：8 个月全部上涨，没有一个月例外

这是本册最硬的一条结论。用同月对比（2016 vs 2017 的 1—8 月），排除季节性。

In [ ]:
# ── 出图 D03_04：同月同比哑铃图 ──
# 为什么用哑铃而不是双折线：8 个月各有一条线段，线段长度就是涨幅，
# 不需要读者在两条线之间竖直比划；哪个月涨得最多一眼能看出来。
# 规范落在哪几行：左端灰、右端主色 / 每个点标出两个数值 / 顺序按月份（有序变量）
# ---- 1 指标 ----
a16 = timing.comparable_slice(d, [2016], a["comparable_months"])
b17 = timing.comparable_slice(d, [2017], a["comparable_months"])
r16 = a16.groupby("arrival_month_no")["is_canceled"].mean().reindex(a["comparable_months"])
r17 = b17.groupby("arrival_month_no")["is_canceled"].mean().reindex(a["comparable_months"])
delta = (r17 - r16)
assert (delta > 0).all(), "有月份没涨，'全部上涨'这句话就不成立"
title = (f"同月对比 {r16.mean():.0%}→{r17.mean():.0%}，{int((delta > 0).sum())} 个月全部上涨，"
         f"{delta.idxmax()} 月涨得最多")
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = (f"口径：只取两年都有的 1—8 月（{len(a16):,} 条 vs {len(b17):,} 条）；"
            f"每点是该月取消率，线段长度即涨幅")
# ---- 2 画图 ----
fig, ax = plt.subplots(figsize=(8.4, 4.4))
viz.dumbbell(ax, [f"{m} 月" for m in a["comparable_months"]],
             r16.to_numpy(), r17.to_numpy(), left_name="2016", right_name="2017")
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D03_04_同月同比哑铃.png")
# ---- 4 读回断言：每根哑铃的两个端点必须是两个时点的取消率 ----
offsets = [c.get_offsets() for c in ax.collections]
assert len(r16) == len(r17) == 8
assert (r17 > r16).all(), "有月份没涨，标题就写错了"
assert delta.idxmax() == 1, f"涨得最多的不是 1 月，而是 {delta.idxmax()} 月"
plt.show()

八个月无例外，说明这不是季节性波动，而是**趋势性上升**：2017 年整体的取消倾向比 2016 年高。
涨幅最大的是 1 月（16.4% → 26.3%，+9.9pp），最小的是 3 月（+2.8pp）。

这条趋势是 D06 的起点：6.4 个百分点的上涨里，有多少是"客人变了"，有多少是"我们的政策变了"？

## 6 本册结论

| 尺度 | 最忙 / 最高 | 最闲 / 最低 | 差距 |
| --- | --- | --- | --- |
| 年内（2016） | 8 月，季节指数 1.27 | 1 月，0.52 | 2.4 倍 |
| 周内（到店量） | 周一 16.2% | 周二 12.5% | 3.7pp |
| 周内（取消率） | 周五 31.0% | 周二 24.7% | 6.3pp |
| 提前期（取消率） | 365 天以上 41.9% | 7 天内 8.4% | **33.5pp** |
| 停留（取消率） | 8 晚以上 33.6% | 1 晚 22.6% | 11.0pp |

**唯一值得拿来建模或做规则的是提前期**：单调、有量、下单时就知道。
季节与星期更适合排班和定价，不适合当取消的抓手。

## 7 发现清单（本册）

| # | 发现 | 影响 |
| --- | --- | --- |
| 1 | 2016 年 8 月与 10 月是双峰 | "旺季"不是一个连续的区间，加人要分两段 |
| 2 | 周内到店量与取消率的峰不在同一天 | 排班看周一，风控看周五，不能合成一个指标 |
| 3 | 提前期与取消率严格单调 | 这是全项目唯一可直接用于规则的变量 |
| 4 | 365 天以上样本仅 547 条 | 它的高取消率有区间但没量，不据此定政策 |
| 5 | 2017 前 8 月每月都在涨 | 趋势性上升，不是季节波动；D06 负责拆来源 |

In [ ]:
con.close()
print("D03 完成：时间四个尺度已量化，提前期是唯一单调变量")